## Deep Neural Network for MNSIT Classification

The dataset is called MNIST and refers to handwritten digit recognition. The dataset provides 70,000 images (28*28 pixels) of handwritten digits (1 digit per image)

The goal is to write an algorithm that detects which digit is written. Since there are only 10 digits (0,1,2,3,4,5,6,7,8,9) this is a classification problem with 10 classes.

Neural network with 2 hidden layers

In [6]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

## Data Processing

In [66]:
mnist_dataset ,mnist_info = tfds.load(name='mnist',with_info=True,as_supervised=True)
mnist_dataset

{'train': <_PrefetchDataset element_spec=(TensorSpec(shape=(28, 28, 1), dtype=tf.uint8, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>,
 'test': <_PrefetchDataset element_spec=(TensorSpec(shape=(28, 28, 1), dtype=tf.uint8, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>}

In [67]:
mnist_info

tfds.core.DatasetInfo(
    name='mnist',
    full_name='mnist/3.0.1',
    description="""
    The MNIST database of handwritten digits.
    """,
    homepage='http://yann.lecun.com/exdb/mnist/',
    data_dir='/Users/shwetha/tensorflow_datasets/mnist/3.0.1',
    file_format=tfrecord,
    download_size=11.06 MiB,
    dataset_size=21.00 MiB,
    features=FeaturesDict({
        'image': Image(shape=(28, 28, 1), dtype=uint8),
        'label': ClassLabel(shape=(), dtype=int64, num_classes=10),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <SplitInfo num_examples=10000, num_shards=1>,
        'train': <SplitInfo num_examples=60000, num_shards=1>,
    },
    citation="""@article{lecun2010mnist,
      title={MNIST handwritten digit database},
      author={LeCun, Yann and Cortes, Corinna and Burges, CJ},
      journal={ATT Labs [Online]. Available: http://yann.lecun.com/exdb/mnist},
      volume={2},
 

In [68]:
mnist_train,mnist_test = mnist_dataset['train'],mnist_dataset['test']

In [69]:
mnist_train

<_PrefetchDataset element_spec=(TensorSpec(shape=(28, 28, 1), dtype=tf.uint8, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [70]:
mnist_test

<_PrefetchDataset element_spec=(TensorSpec(shape=(28, 28, 1), dtype=tf.uint8, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [71]:
num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples

6000.0

In [72]:
num_validation_samples = tf.cast(num_validation_samples,tf.int64)
num_validation_samples

<tf.Tensor: shape=(), dtype=int64, numpy=6000>

In [73]:
num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples

10000

In [74]:
num_test_samples = tf.cast(num_test_samples,tf.int64)
num_test_samples

<tf.Tensor: shape=(), dtype=int64, numpy=10000>

In [75]:
def scaling(image,label):
    image = tf.cast(image,tf.float32)
    image /= 255.  # Signifies we want the result to be in float
    return image,label

In [76]:
scaled_train_validation_data = mnist_train.map(scaling)
scaled_train_validation_data

<_MapDataset element_spec=(TensorSpec(shape=(28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [77]:
test_data = mnist_test.map(scaling)

### Create a batch size of 10000

In [78]:
BUFFER_SIZE = 10000

In [79]:
shuffled_train_validation_data = scaled_train_validation_data.shuffle(BUFFER_SIZE)

In [80]:
shuffled_train_validation_data

<_ShuffleDataset element_spec=(TensorSpec(shape=(28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [81]:
validation_data = shuffled_train_validation_data.take(num_validation_samples)
validation_data

<_TakeDataset element_spec=(TensorSpec(shape=(28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [82]:
train_data = shuffled_train_validation_data.skip(num_validation_samples)
train_data

<_SkipDataset element_spec=(TensorSpec(shape=(28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [83]:
BATCH_SIZE = 100

In [84]:
train_data = train_data.batch(BATCH_SIZE)
train_data

<_BatchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [85]:
validation_data = validation_data.batch(num_validation_samples)
validation_data

<_BatchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>

In [86]:
test_data = test_data.batch(num_test_samples)

In [87]:
validation_inputs , validation_targets = next(iter(validation_data))

W0000 00:00:1791879884.929759 7856854 cache_dataset_ops.cc:912] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [88]:
validation_targets

<tf.Tensor: shape=(6000,), dtype=int64, numpy=array([1, 1, 9, ..., 5, 1, 7], shape=(6000,))>

## Model

## Outline the model

In [89]:
28 * 28

784

In [90]:
input_size = 784
output_size = 10
hidden_layer = 50

In [91]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28,28,1)),
    tf.keras.layers.Dense(hidden_layer,activation='relu'),
    tf.keras.layers.Dense(hidden_layer,activation='relu'),
    tf.keras.layers.Dense(output_size,activation='softmax')
])

## Choose the optimizer and the loss function

In [92]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy']) # Adaptive moment estimation (Adam)

## Training

In [93]:
NUM_EPOCHS = 5

In [94]:
model.fit(train_data,epochs=NUM_EPOCHS,validation_data=(validation_inputs,validation_targets),verbose=2)

Epoch 1/5
540/540 - 1s - 3ms/step - accuracy: 0.8849 - loss: 0.4140 - val_accuracy: 0.9307 - val_loss: 0.2242
Epoch 2/5
540/540 - 1s - 1ms/step - accuracy: 0.9488 - loss: 0.1751 - val_accuracy: 0.9555 - val_loss: 0.1557
Epoch 3/5
540/540 - 1s - 1ms/step - accuracy: 0.9604 - loss: 0.1337 - val_accuracy: 0.9617 - val_loss: 0.1320
Epoch 4/5
540/540 - 1s - 1ms/step - accuracy: 0.9672 - loss: 0.1091 - val_accuracy: 0.9663 - val_loss: 0.1137
Epoch 5/5
540/540 - 1s - 1ms/step - accuracy: 0.9722 - loss: 0.0931 - val_accuracy: 0.9710 - val_loss: 0.0991
